# All LLM based practice

In [1]:
import openai
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
api_key=os.getenv('API_KEY')
base_url=os.getenv('BASE_URL')


In [3]:
message="Hello, I am Hari Vallabh, nice to meet you"
messages=[{"role":"system","content":"You are a helpful assistant"},
          {"role":"user","content":message}
          ]

In [4]:
from openai import OpenAI
openai=OpenAI(api_key=api_key,base_url=base_url)
response=openai.chat.completions.create(model='gpt-6-luna',messages=messages)
response.choices[0].message.content

'Nice to meet you too, Hari Vallabh! How can I help you today?'

### Google-genai model

In [5]:
#gemini api and base url
import os
gemini_api=os.getenv('GEMINI_API_KEY')
gemini_url=os.getenv('GEMINI_BASE_URL')


In [6]:
from google import genai
client=genai.Client(api_key=gemini_api)
response=client.models.generate_content(
    model='gemini-2.5-flash',
    contents="Hello. I am hari vallabh."
)
print(response.text)

Hello, Hari Vallabh! It's a pleasure to meet you.

I'm an AI assistant, ready to help you with whatever you need. How can I assist you today?


### Claude Model

In [62]:
import os
load_dotenv()
claude_api=os.getenv('CLAUDE_API')
claude_url=os.getenv('CLAUDE_BASE_URL')
anthropic=OpenAI(api_key=claude_api,base_url=claude_url)

message="Hello, I am Hari Vallabh, nice to meet you"
messages=[{"role":"system","content":"You are a helpful assistant"},
          {"role":"user","content":message}
          ]
response=anthropic.chat.completions.create(model='anthropic/claude-haiku-5-5',messages=messages)

In [64]:
response.choices[0].message.content

'Hello, Hari Vallabh! Nice to meet you too. How can I help you today?'

## Using Gradio UI interface

In [9]:
import gradio as gr

In [10]:
system_message="You are an helpful assistant"

In [11]:
def message_gpt(prompt):
        messages=[
            {"role":"system","content":system_message},
            {"role":"user","content":prompt}
        ]
        response=openai.chat.completions.create(model='gpt-6-luna',messages=messages)
        return response.choices[0].message.content


In [12]:
def shout(text):
    print(f'Shout has been called {text}')
    return text.upper()

### GPT-Gradio UI

In [13]:
import gradio as gr

inputs=gr.Textbox(label='Your message',info='Enter your message here for GPT-6-Luna',lines=10)
outputs=gr.Markdown(label='Response')

gpt_interface=gr.Interface(
    fn=message_gpt,inputs=inputs,outputs=outputs,examples=['Hello','What is the capital of France?','Who is Elon'],flagging_mode='never',title='GPT'
)


## Using Streaming 

In [17]:
def stream_gpt(prompt):
    messages=[{"role":'system',"comtent":system_message},
              {"role":"user","content":prompt}]
    streaming=openai.chat.completions.create(model='gpt-6-luna',messages=messages,stream=True)
    result=""
    for i in streaming:
        result+=i.choices[0].delta.content or "" 
        yield result

In [19]:
message_input=gr.Textbox(label='Your Message',info='Enter your message here',lines=10)
message_output=gr.Markdown(label='The response')

stemming_ui=gr.Interface(
    fn=stream_gpt,
    title='GPT-6-Luna',
    inputs=message_input,
    outputs=message_output,
    examples=['Explain Transfromer architecture to a newbie','What is GenAI'],
    flagging_mode="never"
)

# Select Multiple models and making an streaming UI

In [74]:
system_message="You are a helpful assistant. You have the capabilities to explain complex topics simply."
def stream_gpt(prompt):
    messages=[{"role":"system","content":system_message},{"role":"user","content":prompt}]
    streaming1=openai.chat.completions.create(model="gpt-6-luna",messages=messages,max_tokens=1000,stream=True)
    result=""
    for i in streaming1:
        if i.choices and i.choices[0].delta.content:
            result+=i.choices[0].delta.content
            yield result
def stream_claude(prompt):
    messages=[{"role":"system","content":system_message},{"role":"user","content":prompt}]
    streaming=anthropic.chat.completions.create(model="anthropic/claude-haiku-5-5",messages=messages,max_tokens=1000,stream=True)
    result=""
    for i in streaming:
        if i.choices and i.choices[0].delta.content:
            result+=i.choices[0].delta.content
            yield result
def stream_model(prompt,model):
    if model=="GPT-6 Luna":
        result=stream_gpt(prompt)
    elif model=="Claude-Haiku 5.5":
        result=stream_claude(prompt)
    else:
        raise ValueError(f"Unknown model:{model}")
    yield from result
message_input=gr.Textbox(label="Your Message",info="Enter your message here",lines=10)
model_select=gr.Dropdown(["GPT-6 Luna","Claude-Haiku 5.5"],label="Select a model",value="GPT-6 Luna")
message_output=gr.Markdown(label="The response")
streaming_ui=gr.Interface(fn=stream_model,title="LLM Models",inputs=[message_input,model_select],outputs=message_output,flagging_mode="never")


# Main Work starts here

### Using multiple models, creating a Chat UI and customer support model.

In [91]:
def chat(message,history):
    history=[{"role":h["role"],"content":h["content"]} for h in history]
    messages=[{"role":"system","content":system_message}]+history+[{"role":"user","content":message}]
    stream=anthropic.chat.completions.create(model='anthropic/claude-haiku-5-5',messages=messages,stream=True)
    result=""
    for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                result+=chunk.choices[0].delta.content
                yield result
    return response.choices[0].message.content

#gr.ChatInterface(fn=chat,type="messages").launch()